In [0]:
from pyspark.sql.functions import col, to_date, year, month, trim

silver_df = (
    spark.table("virginia_energy_bronze")
    .select(
        to_date("period", "yyyy-MM").alias("period_date"),
        "location",
        col("stateDescription").alias("state"),
        col("sectorid").alias("sector_id"),
        col("sectorDescription").alias("sector"),
        col("fueltypeid").alias("fuel_type_id"),
        trim(col("fuelTypeDescription")).alias("fuel_type"),
        col("generation").alias("generation_raw"),
        col("generation").cast("double").alias("generation_mwh"),
        col("generation-units").alias("generation_units"),
        "_ingested_at"
    )
    .withColumn("year", year("period_date"))
    .withColumn("month", month("period_date"))
)

display(silver_df)

period_date,location,state,sector_id,sector,fuel_type_id,fuel_type,generation_raw,generation_mwh,generation_units,_ingested_at,year,month
2025-12-01,VA,Virginia,1,Electric Utility,ALL,all fuels,7068.27934,7068.27934,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,AOR,all renewables,242.04817,242.04817,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,BIO,biomass,111.15839,111.15839,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,BIS,bituminous coal and synthetic coal,458.76735,458.76735,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,BIT,bituminous coal,458.76735,458.76735,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,COL,"coal, excluding waste coal",458.76735,458.76735,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,COW,all coal products,458.76735,458.76735,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,DFO,distillate fuel oil,36.56516,36.56516,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,DPV,estimated small scale solar photovoltaic,0,0.0,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12
2025-12-01,VA,Virginia,1,Electric Utility,FOS,fossil fuels,4727.84399,4727.84399,thousand megawatthours,2026-09-10T06:40:45.779Z,2025,12


In [0]:
print("Null period dates:", silver_df.filter(col("period_date").isNull()).count())
print("Invalid generation values:", silver_df.filter(
    col("generation_mwh").isNull() & col("generation_raw").isNotNull()
).count())
print("Non-Virginia records:", silver_df.filter(col("location") != "VA").count())
print("Null fuel IDs:", silver_df.filter(col("fuel_type_id").isNull()).count())
print("Null generation units:", silver_df.filter(col("generation_units").isNull()).count())

Null period dates: 0
Invalid generation values: 0
Non-Virginia records: 0
Null fuel IDs: 0
Null generation units: 0


In [0]:
from pyspark.sql.functions import count

duplicate_df = (
    silver_df
    .groupBy(
        "period_date",
        "location",
        "sector_id",
        "fuel_type_id"
    )
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

print("Duplicate key groups:", duplicate_df.count())

display(duplicate_df)

Duplicate key groups: 0


period_date,location,sector_id,fuel_type_id,record_count


In [0]:
silver_table = "virginia_energy_silver"

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("Silver table created successfully.")

Silver table created successfully.


In [0]:
spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT period_date) AS distinct_months,
        MIN(period_date) AS earliest_date,
        MAX(period_date) AS latest_date,
        COUNT(DISTINCT fuel_type_id) AS distinct_fuel_types,
        COUNT(DISTINCT sector_id) AS distinct_sectors
    FROM virginia_energy_silver
""").show()

+-------------+---------------+-------------+-----------+-------------------+----------------+
|total_records|distinct_months|earliest_date|latest_date|distinct_fuel_types|distinct_sectors|
+-------------+---------------+-------------+-----------+-------------------+----------------+
|        22610|             72|   2020-01-01| 2025-12-01|                 36|              14|
+-------------+---------------+-------------+-----------+-------------------+----------------+

